# Data Source 05 · 에너지·환경

| 출처 | 접근 | 검증 상태 |
|---|---|---|
| NOAA GML Mauna Loa CO₂ 월별 (`co2_mm_mlo.csv`) | 무키 | A |
| Open-Meteo Historical Weather (ERA5 재분석) | 무키 | A (일평균 변수명은 응답으로 확인) |
| Open-Meteo Air Quality | 무키 | A (한국 지점은 해상도 낮음, 예측 자료는 실측 아님) |
| Open Power System Data / UCI Household Power | 수동 | B / A |
| 기상청 KMA / AirKorea | 무료키·신청 | C |

NOAA 파일은 `#` 주석 행이 있고 결측이 **−99.99**로 표시된다. Open-Meteo는 **관측소 값이 아니라 재분석 격자 값**이다.

In [ ]:
import io, json, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt

def fetch_bytes(url, timeout=30):
    """무키 공개 URL 요청. 실패 시 예외 -> 호출부에서 합성 대체."""
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

def synthetic_series(n, freq, seed, trend=0.05, season=12, noise=1.0, start="2015-01-01"):
    """NOT real data: 방법 시연용 합성 시계열."""
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    y = 100 + trend * t + 3 * np.sin(2 * np.pi * t / season) + rng.normal(0, noise, n)
    return pd.Series(y, index=pd.date_range(start, periods=n, freq=freq), name="value")

def quality_report(s):
    s = s.sort_index()
    print("관측 수:", len(s), "| 기간:", s.index.min(), "~", s.index.max())
    print("결측:", int(s.isna().sum()), "| 중복 시점:", int(s.index.duplicated().sum()))
    f = pd.infer_freq(s.index) if len(s) > 2 else None
    print("추정 주기:", f)

In [ ]:
def parse_noaa_co2(raw):
    text = raw.decode("utf-8") if isinstance(raw, (bytes, bytearray)) else raw
    df = pd.read_csv(io.StringIO(text), comment="#")
    df.columns = [c.strip().lower() for c in df.columns]
    if not {"year", "month", "average"}.issubset(df.columns):
        raise ValueError("예상 열 없음: " + ", ".join(df.columns))
    s = df["average"].where(df["average"] > -90)          # -99.99 -> NaN
    idx = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))
    return pd.Series(s.values, index=idx, name="co2_ppm")

def parse_openmeteo_daily(obj):
    d = obj["daily"]
    tmax = pd.Series(d["temperature_2m_max"], index=pd.to_datetime(d["time"]))
    tmin = pd.Series(d["temperature_2m_min"], index=pd.to_datetime(d["time"]))
    return ((tmax + tmin) / 2).rename("t_mid")             # 최고·최저 평균: 정의된 일평균이 아님

In [ ]:
try:
    co2 = parse_noaa_co2(fetch_bytes("https://gml.noaa.gov/aftp/products/trends/co2/co2_mm_mlo.csv"))
    DATA_KIND = "REAL: NOAA GML Mauna Loa CO2"
except Exception as e:
    print("NOAA 수집 실패 -> 합성 대체:", type(e).__name__, e)
    co2 = synthetic_series(720, "MS", 2026051, trend=0.15, season=12, noise=0.3, start="1965-01-01")
    DATA_KIND = "SYNTHETIC (NOT real data)"
print("DATA_KIND (CO2):", DATA_KIND); quality_report(co2); co2.plot(title=DATA_KIND); plt.show()

# 선택: Open-Meteo 서울 일 기온 (무키, 비상업 무료, 호출 제한 있음)
try:
    u = ("https://archive-api.open-meteo.com/v1/archive?latitude=37.57&longitude=126.98"
         "&start_date=2020-01-01&end_date=2023-12-31&daily=temperature_2m_max,temperature_2m_min&timezone=Asia%2FSeoul")
    t = parse_openmeteo_daily(json.loads(fetch_bytes(u, timeout=60)))
    print("REAL (ERA5 재분석, 관측소 아님):", len(t), "일"); t.plot(title="Seoul t_mid (Open-Meteo/ERA5)"); plt.show()
except Exception as e:
    print("Open-Meteo 실패 (선택 항목):", type(e).__name__, e)

## 점검 체크리스트
- 재분석/모델 값과 관측소 실측을 구분해 보고한다
- 결측 코드(−99.99 등)와 시간대(UTC/현지)
- 예측(forecast) 자료를 과거 실측처럼 쓰지 않는다
- 국내 자료(KMA, AirKorea, KPX)는 키·이용약관 확인 후 추가

## 추가 C. NOAA CO₂ 대체 URL · 일 최저기온 · 조석 (학생 예제 01, 02, 04, 05번)
- **CO₂**: `csv`와 `txt` 두 형식, 여러 URL을 순서대로 시도한다. 평균열 결측은 −99.99이며, 학생 예제 04번의 `{-9.99, -0.99, -1}` 치환은 평균열 결측에 해당하지 않는다(확인 필요). 값 범위 검증(300–600 ppm)을 넣었다.
- **일 최저기온**: Melbourne 1981–1990 일 최저기온(jbrownlee/Datasets). 개인 저장소라 영구성이 보장되지 않는다. 서울 자료가 아니다.
- **조석(Boston 8443970)**: NOAA Tides & Currents 시간별 수위. 주 조석 주기는 약 12.42시간이다. 학생 예제 05번은 계절주기를 12로 둔다.

In [ ]:
def parse_noaa_co2_any(raw):
    """NOAA co2_mm_mlo 의 csv/txt 형식을 모두 처리. 평균열 결측(-99.99)은 NaN."""
    text = raw.decode("utf-8") if isinstance(raw, (bytes, bytearray)) else raw
    lines = [ln for ln in text.splitlines() if ln.strip() and not ln.lstrip().startswith("#")]
    if "," in lines[0]:                                  # csv: 첫 비주석 행이 헤더
        df = pd.read_csv(io.StringIO("\n".join(lines)))
        df.columns = [c.strip().lower() for c in df.columns]
    else:                                                # txt: 공백 구분, 헤더 없음
        cols = ["year", "month", "decimal_date", "average", "deseasonalized", "ndays", "stdev", "uncertainty"]
        df = pd.read_csv(io.StringIO("\n".join(lines)), sep=r"\s+", header=None, names=cols)
    if not {"year", "month", "average"}.issubset(df.columns):
        raise ValueError("예상 열 없음: " + ", ".join(df.columns))
    avg = pd.to_numeric(df["average"], errors="coerce")
    avg = avg.where(avg > -90)
    idx = pd.to_datetime(dict(year=df["year"].astype(int), month=df["month"].astype(int), day=1))
    return pd.Series(avg.values, index=idx, name="co2_ppm")

def check_co2(s):
    v = s.dropna()
    assert len(v) > 600 and v.min() > 300 and v.max() < 600, "CO2 값 범위/길이 이상"

def parse_daily_min_temp(raw):
    df = pd.read_csv(io.BytesIO(raw) if isinstance(raw, (bytes, bytearray)) else io.StringIO(raw))
    df.columns = [c.strip().lower() for c in df.columns]
    return pd.Series(pd.to_numeric(df["temp"], errors="coerce").values, index=pd.to_datetime(df["date"]), name="temp_min")

def parse_tides_json(obj):
    if "data" not in obj:
        raise ValueError("API 응답에 data 없음: " + str(obj)[:120])
    df = pd.DataFrame(obj["data"])
    t = pd.to_datetime(df["t"], errors="coerce")
    return pd.Series(pd.to_numeric(df["v"], errors="coerce").values, index=t, name="water_level_m")[t.notna().values]

def fetch_tides_year(station, year):
    from urllib.parse import urlencode
    q = urlencode(dict(product="hourly_height", application="TimeSeriesCourse", begin_date=f"{year}0101", end_date=f"{year}1231",
                       station=station, datum="MLLW", time_zone="gmt", units="metric", format="json"))
    return parse_tides_json(json.loads(fetch_bytes("https://api.tidesandcurrents.noaa.gov/api/prod/datagetter?" + q, timeout=90)))

In [ ]:
CO2_URLS = ["https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.csv",
            "https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.txt"]
co2b = None
for u in CO2_URLS:
    try:
        co2b = parse_noaa_co2_any(fetch_bytes(u)); check_co2(co2b); KIND_CO2B = "REAL: " + u; break
    except Exception as e:
        print("CO2 시도 실패:", u.split('/')[-1], type(e).__name__, str(e)[:80]); co2b = None
if co2b is None:
    co2b = synthetic_series(816, "MS", 2026081, trend=0.2, season=12, noise=0.3, start="1958-03-01"); KIND_CO2B = "SYNTHETIC (NOT real data)"
print("KIND_CO2B:", KIND_CO2B, "| n =", len(co2b), "| 결측:", int(co2b.isna().sum()))

try:
    tmin = parse_daily_min_temp(fetch_bytes("https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv"))
    assert len(tmin) > 3000 and tmin.dropna().between(-10, 40).all(); KIND_TMIN = "REAL: Melbourne daily min temp 1981-1990"
except Exception as e:
    print("일 최저기온 실패 -> 합성 대체:", type(e).__name__, str(e)[:80])
    t = np.arange(3650); rng = np.random.default_rng(2026082)
    tmin = pd.Series(11 + 5 * np.sin(2 * np.pi * t / 365.25) + rng.normal(0, 2.5, 3650), index=pd.date_range("1981-01-01", periods=3650, freq="D"))
    KIND_TMIN = "SYNTHETIC (NOT real data)"
print("KIND_TMIN:", KIND_TMIN, "| n =", len(tmin))

try:
    tides = pd.concat([fetch_tides_year("8443970", y) for y in (2023, 2024, 2025)]).sort_index()
    tides = tides[~tides.index.duplicated()]
    expected = len(pd.date_range("2023-01-01 00:00", "2025-12-31 23:00", freq="h"))
    print("조석 행:", len(tides), "/ 기대", expected)
    assert len(tides) > 0.95 * expected
    KIND_TIDE = "REAL: NOAA Tides & Currents 8443970 hourly_height (MLLW, GMT)"
except Exception as e:
    print("조석 수집 실패 -> 합성 대체:", type(e).__name__, str(e)[:80])
    h = np.arange(24 * 365); rng = np.random.default_rng(2026083)
    tides = pd.Series(1.5 + 1.4 * np.sin(2 * np.pi * h / 12.42) + 0.3 * np.sin(2 * np.pi * h / 354.4) + rng.normal(0, 0.1, len(h)),
                      index=pd.date_range("2024-01-01", periods=len(h), freq="h"), name="water_level_m")
    KIND_TIDE = "SYNTHETIC (NOT real data)"
print("KIND_TIDE:", KIND_TIDE, "| n =", len(tides))
fig, ax = plt.subplots(3, 1, figsize=(10, 7)); co2b.plot(ax=ax[0], title=KIND_CO2B)
tmin.plot(ax=ax[1], title=KIND_TMIN); tides.iloc[:24 * 14].plot(ax=ax[2], title="조석 첫 14일: " + KIND_TIDE); plt.tight_layout(); plt.show()